# Serebii Champions Scraper
Scrapes move data, pokemon data, ability data, and pokemon images from Serebii's Champions dex.

Outputs:
- `moves_dict.pkl` — dictionary of all moves with type, category, power, effect, priority, targets
- `pokemon_dict.pkl` — dictionary of all pokemon (including mega forms, regional forms, and special multi-forms) with types, abilities, base stats, and move lists
- `ability_dict.pkl` — dictionary of all abilities with descriptions
- `pokemon_images/` — folder containing one PNG per pokemon entry (named to match the keys in pokemon_dict.pkl)

Form naming convention:
- Mega forms: `PokemonName-Mega` (e.g., `Garchomp-Mega`). Charizard has two: `Charizard-Mega-X` and `Charizard-Mega-Y`.
- Alolan forms: `PokemonName-Alola` (e.g., `Ninetales-Alola`)
- Galarian forms: `PokemonName-Galar` (e.g., `Slowbro-Galar`)
- Hisuian forms: `PokemonName-Hisui` (e.g., `Zoroark-Hisui`)
- Paldean forms: `PokemonName-Paldea`. Paldean Tauros has 3 breeds: `Tauros-Paldea-Combat`, `Tauros-Paldea-Blaze`, `Tauros-Paldea-Aqua`.
- Rotom appliance forms: `Rotom-Heat`, `Rotom-Wash`, `Rotom-Frost`, `Rotom-Fan`, `Rotom-Mow`
- Lycanroc forms: base `Lycanroc` (Midday), `Lycanroc-Midnight`, `Lycanroc-Dusk`

In [ ]:
import requests
from bs4 import BeautifulSoup
import pickle
import time
import re
import os

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
HEADERS = {'User-Agent': 'Mozilla/5.0'}
BASE_URL = 'https://www.serebii.net'

## Part 1: Scrape All Moves

In [ ]:
# Collect all move URLs from the attackdex index page
resp = requests.get(f'{BASE_URL}/attackdex-champions/', timeout=15, headers=HEADERS)
soup = BeautifulSoup(resp.text, 'html.parser')

move_urls = set()
for select in soup.find_all('select', {'name': 'SelectURL'}):
    for option in select.find_all('option'):
        val = option.get('value', '').strip()
        if val and val.endswith('.shtml'):
            move_urls.add(val)

move_urls = sorted(move_urls)
print(f'Found {len(move_urls)} move URLs to scrape')

In [ ]:
def parse_move_page(soup):
    """Parse a move page and return (move_name, info_dict, has_pokemon)."""
    dextables = soup.find_all('table', class_='dextable')
    if not dextables:
        return None, None, False

    dt = dextables[0]
    rows = dt.find_all('tr')

    move_name = None
    battle_type = None
    category = None
    power = None
    effect = None
    priority = None
    targets = None

    for i, r in enumerate(rows):
        cells = r.find_all('td')
        texts = [c.get_text(strip=True) for c in cells]

        # Row with move name, type image, category image
        if len(cells) == 3:
            type_imgs = [img['src'] for img in cells[1].find_all('img') if '/type/' in img.get('src', '')]
            cat_imgs = [img['src'] for img in cells[2].find_all('img') if '/type/' in img.get('src', '')]

            if type_imgs and not move_name:
                # Extract move name (remove Japanese text)
                raw_name = cells[0].get_text(strip=True)
                # Japanese chars start after the English name
                # Use the first <br> or just strip non-ASCII
                # The English name is before any Japanese characters
                match = re.match(r'^([A-Za-z0-9,\-\' \!\?\.]+)', raw_name)
                if match:
                    move_name = match.group(1).strip()
                else:
                    move_name = raw_name

                # Parse type from image filename
                type_match = re.search(r'/type/(\w+)\.', type_imgs[0])
                if type_match:
                    battle_type = type_match.group(1).capitalize()

                # Parse category from image filename
                cat_match = re.search(r'/type/(\w+)\.', cat_imgs[0]) if cat_imgs else None
                if cat_match:
                    cat_val = cat_match.group(1).lower()
                    if cat_val == 'physical':
                        category = 'Physical'
                    elif cat_val == 'special':
                        category = 'Special'
                    elif cat_val == 'other':
                        category = 'Status'

            # Row with Base Power (after Power Points / Base Power / Accuracy header)
            if texts == ['Power Points', 'Base Power', 'Accuracy']:
                # Next row has the values
                if i + 1 < len(rows):
                    val_cells = rows[i + 1].find_all('td')
                    if len(val_cells) == 3:
                        power_text = val_cells[1].get_text(strip=True)
                        try:
                            power = int(power_text)
                        except ValueError:
                            power = 0  # status moves show '--' or '0'

            # Row with priority and targets
            if texts == ['Base Critical Hit Rate', 'Speed Priority', 'Pokémon Hit in Battle']:
                if i + 1 < len(rows):
                    val_cells = rows[i + 1].find_all('td')
                    if len(val_cells) == 3:
                        try:
                            priority = int(val_cells[1].get_text(strip=True))
                        except ValueError:
                            priority = 0
                        targets = val_cells[2].get_text(strip=True)

        # Battle Effect row (colspan=1, single cell with the effect text)
        if len(cells) == 1 and texts and 'Battle Effect:' not in texts[0] and 'Secondary Effect:' not in texts[0]:
            # This could be the effect description
            # It comes right after 'Battle Effect:' header
            if i > 0:
                prev_cells = rows[i - 1].find_all('td')
                prev_texts = [c.get_text(strip=True) for c in prev_cells]
                if prev_texts == ['Battle Effect:']:
                    effect = texts[0]

    # Check if pokemon can learn this move (dextable[2] exists with pokemon rows)
    has_pokemon = False
    if len(dextables) > 2:
        pokemon_table = dextables[2]
        pokemon_rows = pokemon_table.find_all('tr')
        for pr in pokemon_rows[1:]:
            if re.search(r'#\d{4}', pr.get_text()):
                has_pokemon = True
                break

    if not move_name:
        return None, None, False

    info = {
        'type': battle_type,
        'category': category,
        'power': power if power else 0,
        'secondary_effect': effect if effect else 'None',
        'priority': priority if priority is not None else 0,
        'targets': targets if targets else 'Unknown'
    }

    return move_name, info, has_pokemon

In [ ]:
moves_dict = {}
skipped_no_pokemon = []
failed_moves = []

for i, url_path in enumerate(move_urls):
    if (i + 1) % 50 == 0 or i == 0:
        print(f'Scraping move {i + 1}/{len(move_urls)}: {url_path}', flush=True)
    try:
        resp = requests.get(f'{BASE_URL}{url_path}', timeout=15, headers=HEADERS)
        if resp.status_code == 404:
            failed_moves.append((url_path, '404'))
            continue
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        move_name, info, has_pokemon = parse_move_page(soup)

        if move_name and has_pokemon:
            moves_dict[move_name] = info
        elif move_name:
            skipped_no_pokemon.append(move_name)
        else:
            failed_moves.append((url_path, 'parse failed'))
    except Exception as e:
        failed_moves.append((url_path, str(e)))
    time.sleep(0.3)

print(f'\nDone! {len(moves_dict)} moves in dictionary.')
print(f'Skipped (no pokemon learn them): {len(skipped_no_pokemon)}')
if failed_moves:
    print(f'Failed: {len(failed_moves)}')
    for url, reason in failed_moves[:10]:
        print(f'  {url} — {reason}')

In [ ]:
# Preview some moves
for name in ['Flamethrower', 'Fake Out', 'Will-O-Wisp', 'Protect', 'Earthquake']:
    if name in moves_dict:
        print(f'{name}: {moves_dict[name]}')
    else:
        print(f'{name}: NOT FOUND')

In [ ]:
moves_path = os.path.join(NOTEBOOK_DIR, 'moves_dict.pkl')
with open(moves_path, 'wb') as f:
    pickle.dump(moves_dict, f)
print(f'Saved {len(moves_dict)} moves to {moves_path}')

## Part 2: Scrape All Pokemon + Abilities

In [ ]:
# Collect all pokemon URLs from the pokedex index page
resp = requests.get(f'{BASE_URL}/pokedex-champions/', timeout=15, headers=HEADERS)
soup = BeautifulSoup(resp.text, 'html.parser')

pokemon_urls = set()
for s in soup.find_all('select', {'name': 'SelectURL'}):
    for o in s.find_all('option'):
        val = o.get('value', '').strip()
        if val and val.startswith('/pokedex-champions/') and val != '#' and not val.endswith('.shtml'):
            pokemon_urls.add(val)

pokemon_urls = sorted(pokemon_urls)
print(f'Found {len(pokemon_urls)} pokemon URLs to scrape')

In [ ]:
# ---- Pokemon page parser (handles base, mega forms, AND regional forms) ----

def parse_abilities_with_forms(ab_text):
    """Parse abilities text that may contain '(Form Name)' markers.

    Returns dict: form_marker -> [ability_names].
    If no form markers, returns {'_base_': [abilities]}.
    """
    text = re.sub(r'^\s*Abilities:\s*', '', ab_text)
    text = text.split('\n')[0].strip()
    pattern = re.compile(r'\(([^)]+)\)')
    matches = list(pattern.finditer(text))
    if not matches:
        ability_names = [a.strip() for a in re.split(r'\s*-\s*', text) if a.strip()]
        return {'_base_': ability_names}
    forms = {}
    last_end = 0
    for m in matches:
        abs_text = text[last_end:m.start()].strip()
        form_name = m.group(1).strip()
        ability_names = [a.strip() for a in re.split(r'\s*-\s*', abs_text) if a.strip()]
        forms[form_name] = ability_names
        last_end = m.end()
    return forms


def parse_stats_from_table(dt):
    """Extract base stats dict from a stats dextable."""
    for r in dt.find_all('tr'):
        cells = r.find_all('td')
        texts = [c.get_text(strip=True) for c in cells]
        if len(texts) == 7 and 'Base Stats' in texts[0]:
            try:
                return {
                    'hp': int(texts[1]), 'attack': int(texts[2]),
                    'defense': int(texts[3]), 'sp_attack': int(texts[4]),
                    'sp_defense': int(texts[5]), 'speed': int(texts[6])
                }
            except ValueError:
                pass
    return {}


def parse_moves_from_table(dt):
    """Extract list of move names from a moves dextable."""
    moves = []
    for r in dt.find_all('tr'):
        cells = r.find_all('td')
        if len(cells) >= 5:
            type_imgs = [img for img in cells[1].find_all('img') if '/type/' in img.get('src', '')]
            if type_imgs:
                name = cells[0].get_text(strip=True)
                if name:
                    moves.append(name)
    return moves


def parse_types_per_form(name_dt):
    """Parse types from the name/type dextable.

    Returns {form_label: [types]} for pokemon with multiple forms in the name table,
    or {'_base_': [types]} for single-form pokemon.
    """
    type_cells = []
    for tr in name_dt.find_all('tr', recursive=False):
        for c in tr.find_all('td', recursive=False):
            type_imgs = [img for img in c.find_all('img') if '/type/' in img.get('src', '')]
            if type_imgs:
                type_cells.append(c)
    if not type_cells:
        for c in name_dt.find_all('td'):
            type_imgs = [img for img in c.find_all('img') if '/type/' in img.get('src', '')]
            if type_imgs:
                type_cells.append(c)
                break
    if not type_cells:
        return {'_base_': []}

    main_cell = type_cells[0]
    nested_table = main_cell.find('table')
    if nested_table:
        result = {}
        for tr in nested_table.find_all('tr'):
            cells = tr.find_all('td')
            if len(cells) >= 2:
                form_label = cells[0].get_text(strip=True)
                types = []
                for img in cells[1].find_all('img'):
                    src = img.get('src', '')
                    if '/type/' in src:
                        m = re.search(r'/type/(\w+)\.', src)
                        if m:
                            t = m.group(1).capitalize()
                            if t not in types:
                                types.append(t)
                if form_label and types:
                    result[form_label] = types
        if result:
            return result

    # Single form
    types = []
    for img in main_cell.find_all('img'):
        src = img.get('src', '')
        if '/type/' in src:
            m = re.search(r'/type/(\w+)\.', src)
            if m:
                t = m.group(1).capitalize()
                if t not in types:
                    types.append(t)
    return {'_base_': types}


def find_form_abilities(forms_dict, region_label):
    """Find abilities for a regional form matching the given label."""
    region_lower = region_label.lower()
    for marker, abilities in forms_dict.items():
        if region_lower in marker.lower():
            return abilities
    return []


def find_form_types(types_dict, type_label):
    """Find types for a form whose label contains the given keyword."""
    label_lower = type_label.lower()
    for marker, types in types_dict.items():
        if label_lower in marker.lower():
            return types
    return []


def parse_pokemon_page(soup):
    """Parse a pokemon page. Returns:
      - list of (name, info_dict, image_url) tuples (base + megas + regional forms)
      - dict of {ability_name: description}
    """
    dextables = soup.find_all('table', class_='dextable')
    if not dextables:
        return [], {}

    results = []
    abilities_found = {}

    # ---- Categorize tables ----
    name_table_idx = None
    ability_table_idx = None
    base_moves_idx = None
    base_stats_idx = None
    alt_forms_idx = None
    regional_moves = {}    # region_key -> table_idx
    regional_stats = {}    # region_key -> table_idx
    mega_sections = []     # list of {'header_idx': i}

    for i, dt in enumerate(dextables):
        text = dt.get_text(strip=True)
        if 'NameOther Names' in text and 'Gender Ratio' in text and name_table_idx is None:
            name_table_idx = i
        elif text.startswith('Abilities:') and ability_table_idx is None:
            ability_table_idx = i
        elif text.startswith('Alternate Forms') and alt_forms_idx is None:
            alt_forms_idx = i
        elif text.startswith('Standard Moves') and 'Standard Moves -' not in text[:30]:
            if base_moves_idx is None:
                base_moves_idx = i
        elif 'Form Standard Moves' in text[:50]:
            for region_text, region_key in [
                ('Alolan Form', 'Alola'), ('Alola Form', 'Alola'),
                ('Galarian Form', 'Galar'), ('Galar Form', 'Galar'),
                ('Hisuian Form', 'Hisui'), ('Hisui Form', 'Hisui'),
                ('Paldean Form', 'Paldea-Combat'), ('Paldea Form', 'Paldea-Combat'),
            ]:
                if text.startswith(region_text):
                    regional_moves[region_key] = i
                    break
        elif text.startswith('Standard Moves -'):
            head = text[:60]
            if 'Blaze Breed' in head:
                regional_moves['Paldea-Blaze'] = i
            elif 'Aqua Breed' in head:
                regional_moves['Paldea-Aqua'] = i
            elif 'Combat Breed' in head:
                regional_moves['Paldea-Combat'] = i
        elif text.startswith('Stats') and 'Base Stats' in text and not text.startswith('Stats - '):
            if base_stats_idx is None:
                base_stats_idx = i
        elif text.startswith('Stats - ') and 'Base Stats' in text:
            head = text[:80]
            if 'Alolan' in head or 'Alola' in head:
                regional_stats['Alola'] = i
            elif 'Galarian' in head or 'Galar' in head:
                regional_stats['Galar'] = i
            elif 'Hisuian' in head or 'Hisui' in head:
                regional_stats['Hisui'] = i
            elif 'Paldean' in head or 'Paldea' in head:
                regional_stats['Paldea'] = i
        elif re.match(r'^Mega\s+', text) and 'Picture' in text:
            mega_sections.append({'header_idx': i})

    if name_table_idx is None:
        return [], {}

    # ---- Base pokemon name ----
    dt_name = dextables[name_table_idx]
    name_text = dt_name.get_text(strip=True)
    name_match = re.search(r'Type(.+?)Japan:', name_text)
    pokemon_name = name_match.group(1).strip() if name_match else 'Unknown'

    # ---- Types per form ----
    types_per_form = parse_types_per_form(dt_name)
    if '_base_' in types_per_form:
        base_types = types_per_form['_base_']
    else:
        base_types = list(types_per_form.values())[0]

    # ---- Abilities ----
    base_abilities = []
    ab_forms = {}
    if ability_table_idx is not None:
        dt_ab = dextables[ability_table_idx]
        first_td = dt_ab.find('td')
        ab_first_line = first_td.get_text() if first_td else ''
        ab_forms = parse_abilities_with_forms(ab_first_line)

        full_ab_text = dt_ab.get_text()
        all_ab_names = []
        seen = set()
        for a in dt_ab.find_all('a'):
            if '/abilitydex' in a.get('href', ''):
                an = a.get_text(strip=True)
                if an not in seen:
                    all_ab_names.append(an)
                    seen.add(an)
        for ab_name in all_ab_names:
            if ab_name in abilities_found:
                continue
            other_abs = [n for n in all_ab_names if n != ab_name]
            if other_abs:
                pattern = re.escape(ab_name) + r':\s*(.+?)(?=' + '|'.join(re.escape(n) + ':' for n in other_abs) + r')'
            else:
                pattern = re.escape(ab_name) + r':\s*(.+?)$'
            desc_match = re.search(pattern, full_ab_text, re.DOTALL)
            if desc_match:
                desc = re.sub(r'\s+', ' ', desc_match.group(1)).strip()
                abilities_found[ab_name] = desc

        if '_base_' in ab_forms:
            base_abilities = ab_forms['_base_']
        else:
            base_abilities = list(ab_forms.values())[0]

    # ---- Base stats & moves ----
    base_stats = parse_stats_from_table(dextables[base_stats_idx]) if base_stats_idx is not None else {}
    move_names = parse_moves_from_table(dextables[base_moves_idx]) if base_moves_idx is not None else []

    # ---- Base image ----
    pic_table = dextables[0]
    home_imgs = [img['src'] for img in pic_table.find_all('img')
                 if '/pokemonhome/pokemon/' in img.get('src', '')]
    base_image = home_imgs[0] if home_imgs else None

    base_info = {
        'type': base_types if len(base_types) > 1 else (base_types[0] if base_types else 'Unknown'),
        'ability': base_abilities if len(base_abilities) > 1 else (base_abilities[0] if base_abilities else 'Unknown'),
        'hp': base_stats.get('hp', 0), 'attack': base_stats.get('attack', 0),
        'defense': base_stats.get('defense', 0), 'sp_attack': base_stats.get('sp_attack', 0),
        'sp_defense': base_stats.get('sp_defense', 0), 'speed': base_stats.get('speed', 0),
        'moves': move_names
    }
    results.append((pokemon_name, base_info, base_image))

    # ---- Regional forms ----
    region_images = {}
    if alt_forms_idx is not None:
        dt_alt = dextables[alt_forms_idx]
        for src in [img['src'] for img in dt_alt.find_all('img')
                    if '/pokemonhome/pokemon/' in img.get('src', '')]:
            m = re.search(r'\d+(?:-(\w+))?\.png', src)
            if m:
                suffix = m.group(1)
                if suffix == 'a':
                    if 'Alola' in regional_stats:
                        region_images['Alola'] = src
                    elif 'Paldea' in regional_stats:
                        region_images['Paldea-Aqua'] = src
                elif suffix == 'g':
                    region_images['Galar'] = src
                elif suffix == 'h':
                    region_images['Hisui'] = src
                elif suffix == 'p':
                    region_images['Paldea-Combat'] = src
                elif suffix == 'b':
                    region_images['Paldea-Blaze'] = src

    regional_form_keys = []
    if 'Alola' in regional_stats:
        regional_form_keys.append('Alola')
    if 'Galar' in regional_stats:
        regional_form_keys.append('Galar')
    if 'Hisui' in regional_stats:
        regional_form_keys.append('Hisui')
    if 'Paldea' in regional_stats:
        # Tauros: 3 breeds; otherwise just one
        if 'Paldea-Blaze' in regional_moves or 'Paldea-Aqua' in regional_moves:
            regional_form_keys.extend(['Paldea-Combat', 'Paldea-Blaze', 'Paldea-Aqua'])
        else:
            regional_form_keys.append('Paldea')

    for region_key in regional_form_keys:
        if region_key == 'Paldea-Combat':
            suffix_str, stats_key, type_label, ab_label = '-Paldea-Combat', 'Paldea', 'Paldean', 'Paldean'
        elif region_key == 'Paldea-Blaze':
            suffix_str, stats_key, type_label, ab_label = '-Paldea-Blaze', 'Paldea', 'Blaze', 'Paldean'
        elif region_key == 'Paldea-Aqua':
            suffix_str, stats_key, type_label, ab_label = '-Paldea-Aqua', 'Paldea', 'Aqua', 'Paldean'
        elif region_key == 'Paldea':
            suffix_str, stats_key, type_label, ab_label = '-Paldea', 'Paldea', 'Paldean', 'Paldean'
        else:
            suffix_str = f'-{region_key}'
            stats_key = region_key
            type_label = region_key
            ab_label = region_key

        full_name = pokemon_name + suffix_str
        form_stats = parse_stats_from_table(dextables[regional_stats[stats_key]])

        if region_key in regional_moves:
            form_moves = parse_moves_from_table(dextables[regional_moves[region_key]])
        elif stats_key in regional_moves:
            form_moves = parse_moves_from_table(dextables[regional_moves[stats_key]])
        else:
            form_moves = move_names

        form_abilities = find_form_abilities(ab_forms, ab_label) or base_abilities
        form_types = find_form_types(types_per_form, type_label) or base_types

        info = {
            'type': form_types if len(form_types) > 1 else (form_types[0] if form_types else 'Unknown'),
            'ability': form_abilities if len(form_abilities) > 1 else (form_abilities[0] if form_abilities else 'Unknown'),
            'hp': form_stats.get('hp', 0), 'attack': form_stats.get('attack', 0),
            'defense': form_stats.get('defense', 0), 'sp_attack': form_stats.get('sp_attack', 0),
            'sp_defense': form_stats.get('sp_defense', 0), 'speed': form_stats.get('speed', 0),
            'moves': form_moves
        }
        results.append((full_name, info, region_images.get(region_key)))

    # ---- Mega forms ----
    for mega in mega_sections:
        header_idx = mega['header_idx']
        header_text = dextables[header_idx].get_text(strip=True)
        mega_name_match = re.match(r'(Mega\s+.+?)Picture', header_text)
        if not mega_name_match:
            continue
        mega_display = mega_name_match.group(1).strip()
        xy_match = re.match(r'Mega\s+(.+?)\s*([XY])$', mega_display)
        if xy_match:
            mega_key = f"{xy_match.group(1).strip()}-Mega-{xy_match.group(2)}"
        else:
            base_match = re.match(r'Mega\s+(.+)', mega_display)
            mega_key = f"{base_match.group(1).strip()}-Mega" if base_match else mega_display

        mega_image = None
        mega_imgs = [img['src'] for img in dextables[header_idx].find_all('img')
                     if '/legendsz' in img.get('src', '') or '/pokemonhome' in img.get('src', '')]
        if mega_imgs:
            mega_image = mega_imgs[0]

        mega_ability_idx = None
        mega_stats_idx = None
        mega_name_idx = header_idx + 1
        for j in range(header_idx + 1, min(header_idx + 7, len(dextables))):
            t = dextables[j].get_text(strip=True)
            if t.startswith('Abilities:') and mega_ability_idx is None:
                mega_ability_idx = j
            elif 'Base Stats' in t and 'HP' in t and mega_stats_idx is None:
                mega_stats_idx = j

        mega_types = []
        if mega_name_idx < len(dextables):
            mt = parse_types_per_form(dextables[mega_name_idx])
            if '_base_' in mt:
                mega_types = mt['_base_']
            elif mt:
                mega_types = list(mt.values())[0]

        mega_abilities = []
        if mega_ability_idx is not None:
            dt_mab = dextables[mega_ability_idx]
            first_td_mab = dt_mab.find('td')
            mab_text = first_td_mab.get_text() if first_td_mab else ''
            mab_forms = parse_abilities_with_forms(mab_text)
            if '_base_' in mab_forms:
                mega_abilities = mab_forms['_base_']
            elif mab_forms:
                mega_abilities = next(iter(mab_forms.values()))

            full_text = dt_mab.get_text()
            for a in dt_mab.find_all('a'):
                if '/abilitydex' in a.get('href', ''):
                    an = a.get_text(strip=True)
                    if an not in abilities_found:
                        pattern = re.escape(an) + r':\s*(.+?)$'
                        dm = re.search(pattern, full_text, re.DOTALL)
                        if dm:
                            abilities_found[an] = re.sub(r'\s+', ' ', dm.group(1)).strip()

        mega_stats = parse_stats_from_table(dextables[mega_stats_idx]) if mega_stats_idx is not None else {}

        mega_info = {
            'type': mega_types if len(mega_types) > 1 else (mega_types[0] if mega_types else 'Unknown'),
            'ability': mega_abilities if len(mega_abilities) > 1 else (mega_abilities[0] if mega_abilities else 'Unknown'),
            'hp': mega_stats.get('hp', base_stats.get('hp', 0)),
            'attack': mega_stats.get('attack', 0),
            'defense': mega_stats.get('defense', 0),
            'sp_attack': mega_stats.get('sp_attack', 0),
            'sp_defense': mega_stats.get('sp_defense', 0),
            'speed': mega_stats.get('speed', 0),
            'moves': move_names
        }
        results.append((mega_key, mega_info, mega_image))

    return results, abilities_found


def download_image(image_url, save_path):
    """Download an image and save to save_path. Returns True on success."""
    if not image_url:
        return False
    if os.path.exists(save_path):
        return True  # Already downloaded
    full_url = BASE_URL + image_url if image_url.startswith('/') else image_url
    try:
        r = requests.get(full_url, timeout=15, headers=HEADERS)
        if r.status_code == 200:
            with open(save_path, 'wb') as f:
                f.write(r.content)
            return True
    except Exception:
        pass
    return False

In [ ]:
# Set up image directory
IMAGE_DIR = os.path.join(NOTEBOOK_DIR, 'pokemon_images')
os.makedirs(IMAGE_DIR, exist_ok=True)

pokemon_dict = {}
ability_dict = {}
failed_pokemon = []
failed_images = []

for i, url_path in enumerate(pokemon_urls):
    if (i + 1) % 20 == 0 or i == 0:
        print(f'Scraping pokemon {i + 1}/{len(pokemon_urls)}: {url_path}', flush=True)
    try:
        resp = requests.get(f'{BASE_URL}{url_path}', timeout=15, headers=HEADERS)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, 'html.parser')
        entries, ab_dict = parse_pokemon_page(soup)

        for name, info, image_url in entries:
            pokemon_dict[name] = info
            # Download image (named to match the dict key)
            save_path = os.path.join(IMAGE_DIR, f'{name}.png')
            if not download_image(image_url, save_path):
                failed_images.append((name, image_url))

        ability_dict.update(ab_dict)

    except Exception as e:
        failed_pokemon.append((url_path, str(e)))
    time.sleep(0.3)

print(f'\nDone! {len(pokemon_dict)} pokemon entries (including megas and regional forms).')
print(f'Abilities collected: {len(ability_dict)}')
print(f'Images downloaded: {len(pokemon_dict) - len(failed_images)}/{len(pokemon_dict)}')

if failed_pokemon:
    print(f'\nFailed pokemon pages: {len(failed_pokemon)}')
    for url, reason in failed_pokemon[:10]:
        print(f'  {url} — {reason}')

if failed_images:
    print(f'\nFailed images: {len(failed_images)}')
    for name, url in failed_images[:10]:
        print(f'  {name} — {url}')

In [ ]:
# ---- Special multi-form pokemon: Rotom (5 appliance forms) + Lycanroc (Midnight, Dusk) ----
# These pokemon have unusual form structures that don't fit the regional/mega pattern,
# so they're handled separately.

# --- ROTOM (5 appliance forms) ---
print("Scraping Rotom appliance forms...")
resp = requests.get(f'{BASE_URL}/pokedex-champions/rotom/', timeout=15, headers=HEADERS)
soup = BeautifulSoup(resp.text, 'html.parser')
dextables = soup.find_all('table', class_='dextable')

rotom_base_moves_idx = rotom_special_idx = rotom_alt_stats_idx = None
for i, dt in enumerate(dextables):
    text = dt.get_text(strip=True)
    if text.startswith('Standard Moves') and 'Standard Moves -' not in text[:30]:
        rotom_base_moves_idx = i
    elif text.startswith('Special Moves'):
        rotom_special_idx = i
    elif text.startswith('Stats - Alternate Forms'):
        rotom_alt_stats_idx = i

rotom_base_moves = parse_moves_from_table(dextables[rotom_base_moves_idx])
rotom_alt_stats = parse_stats_from_table(dextables[rotom_alt_stats_idx])

# Parse signature moves from Special Moves table
rotom_sigs = {}  # e.g. 'Heat' -> 'Overheat'
last_move = None
for r in dextables[rotom_special_idx].find_all('tr'):
    cells = r.find_all('td')
    if len(cells) >= 5:
        type_imgs = [img for img in cells[1].find_all('img') if '/type/' in img.get('src', '')]
        if type_imgs:
            n = cells[0].get_text(strip=True)
            if n:
                last_move = n
    for t in [c.get_text(strip=True) for c in cells]:
        m = re.match(r'^(\w+) Rotom Only$', t)
        if m and last_move:
            rotom_sigs[m.group(1)] = last_move
            last_move = None

# Rotom forms: (name, image_suffix, types, signature_move)
ROTOM_FORMS = [
    ('Rotom-Heat',  'h', ['Electric', 'Fire'],   rotom_sigs.get('Heat')),
    ('Rotom-Wash',  'w', ['Electric', 'Water'],  rotom_sigs.get('Wash')),
    ('Rotom-Frost', 's', ['Electric', 'Ice'],    rotom_sigs.get('Frost')),
    ('Rotom-Fan',   'f', ['Electric', 'Flying'], rotom_sigs.get('Fan')),
    ('Rotom-Mow',   'm', ['Electric', 'Grass'],  rotom_sigs.get('Mow')),
]

for form_name, img_suffix, types, sig_move in ROTOM_FORMS:
    moves = list(rotom_base_moves)
    if sig_move and sig_move not in moves:
        moves.append(sig_move)
    pokemon_dict[form_name] = {
        'type': types,
        'ability': 'Levitate',
        'hp': rotom_alt_stats.get('hp', 0),
        'attack': rotom_alt_stats.get('attack', 0),
        'defense': rotom_alt_stats.get('defense', 0),
        'sp_attack': rotom_alt_stats.get('sp_attack', 0),
        'sp_defense': rotom_alt_stats.get('sp_defense', 0),
        'speed': rotom_alt_stats.get('speed', 0),
        'moves': moves,
    }
    download_image(f'/pokemonhome/pokemon/479-{img_suffix}.png',
                   os.path.join(IMAGE_DIR, f'{form_name}.png'))
    print(f"  {form_name}: type={types}, moves={len(moves)}")

# --- LYCANROC (Midnight + Dusk forms) ---
print("\nScraping Lycanroc Midnight + Dusk forms...")
resp = requests.get(f'{BASE_URL}/pokedex-champions/lycanroc/', timeout=15, headers=HEADERS)
soup = BeautifulSoup(resp.text, 'html.parser')
dextables = soup.find_all('table', class_='dextable')

lyc_midnight_moves_idx = lyc_dusk_moves_idx = None
lyc_midnight_stats_idx = lyc_dusk_stats_idx = lyc_ab_idx = None
for i, dt in enumerate(dextables):
    text = dt.get_text(strip=True)
    if text.startswith('Standard Moves - Midnight Form'):
        lyc_midnight_moves_idx = i
    elif text.startswith('Standard Moves - Dusk Form'):
        lyc_dusk_moves_idx = i
    elif text.startswith('Stats - Midnight Form'):
        lyc_midnight_stats_idx = i
    elif text.startswith('Stats - Dusk Form'):
        lyc_dusk_stats_idx = i
    elif text.startswith('Abilities:'):
        lyc_ab_idx = i

# Parse abilities per form (Midday, Midnight, Dusk)
ab_text = dextables[lyc_ab_idx].find('td').get_text().split('\n')[0]
ab_forms_lyc = parse_abilities_with_forms(ab_text)

LYCANROC_FORMS = [
    ('Lycanroc-Midnight', 'm', lyc_midnight_stats_idx, lyc_midnight_moves_idx, 'Midnight'),
    ('Lycanroc-Dusk',     'd', lyc_dusk_stats_idx,     lyc_dusk_moves_idx,     'Dusk'),
]

for form_name, img_suffix, stats_idx, moves_idx, marker in LYCANROC_FORMS:
    stats = parse_stats_from_table(dextables[stats_idx])
    moves = parse_moves_from_table(dextables[moves_idx])
    abilities = next((v for k, v in ab_forms_lyc.items() if marker in k), [])
    pokemon_dict[form_name] = {
        'type': 'Rock',
        'ability': abilities if len(abilities) > 1 else (abilities[0] if abilities else 'Unknown'),
        'hp': stats.get('hp', 0),
        'attack': stats.get('attack', 0),
        'defense': stats.get('defense', 0),
        'sp_attack': stats.get('sp_attack', 0),
        'sp_defense': stats.get('sp_defense', 0),
        'speed': stats.get('speed', 0),
        'moves': moves,
    }
    download_image(f'/pokemonhome/pokemon/745-{img_suffix}.png',
                   os.path.join(IMAGE_DIR, f'{form_name}.png'))
    print(f"  {form_name}: ability={pokemon_dict[form_name]['ability']}, moves={len(moves)}")

# Fix base Lycanroc abilities to Midday only (the generic parser grabs the first form's abilities,
# but for Lycanroc that's "Midday Form" already labeled, so this is just defensive cleanup)
if 'Lycanroc' in pokemon_dict:
    midday_abs = next((v for k, v in ab_forms_lyc.items() if 'Midday' in k), [])
    if midday_abs:
        pokemon_dict['Lycanroc']['ability'] = midday_abs if len(midday_abs) > 1 else midday_abs[0]
        print(f"  Updated base Lycanroc ability to: {pokemon_dict['Lycanroc']['ability']}")

print(f"\nTotal pokemon entries after adding special forms: {len(pokemon_dict)}")

In [ ]:
# Preview pokemon (including mega + regional forms)
for name in ['Charizard', 'Charizard-Mega-X', 'Charizard-Mega-Y',
             'Garchomp', 'Garchomp-Mega',
             'Ninetales', 'Ninetales-Alola',
             'Slowbro', 'Slowbro-Galar', 'Slowbro-Mega',
             'Zoroark', 'Zoroark-Hisui',
             'Tauros', 'Tauros-Paldea-Combat', 'Tauros-Paldea-Blaze', 'Tauros-Paldea-Aqua']:
    if name in pokemon_dict:
        info = pokemon_dict[name]
        print(f"\n{name}:")
        print(f"  Type: {info['type']}")
        print(f"  Ability: {info['ability']}")
        print(f"  Stats: HP={info['hp']} Atk={info['attack']} Def={info['defense']} SpA={info['sp_attack']} SpD={info['sp_defense']} Spe={info['speed']}")
        print(f"  Moves ({len(info['moves'])}): {info['moves'][:5]}...")
    else:
        print(f'{name}: NOT FOUND')

In [ ]:
# Preview abilities
for name in ['Blaze', 'Solar Power', 'Tough Claws', 'Drought', 'Defiant', 'Rough Skin']:
    if name in ability_dict:
        print(f"{name}: {ability_dict[name][:100]}")
    else:
        print(f"{name}: NOT FOUND")

In [ ]:
# Save pokemon dict
pokemon_path = os.path.join(NOTEBOOK_DIR, 'pokemon_dict.pkl')
with open(pokemon_path, 'wb') as f:
    pickle.dump(pokemon_dict, f)
print(f'Saved {len(pokemon_dict)} pokemon to {pokemon_path}')

# Save ability dict
ability_path = os.path.join(NOTEBOOK_DIR, 'ability_dict.pkl')
with open(ability_path, 'wb') as f:
    pickle.dump(ability_dict, f)
print(f'Saved {len(ability_dict)} abilities to {ability_path}')

In [ ]:
# Verify all files reload correctly
for fname, label in [('moves_dict.pkl', 'moves'), ('pokemon_dict.pkl', 'pokemon'), ('ability_dict.pkl', 'abilities')]:
    fpath = os.path.join(NOTEBOOK_DIR, fname)
    with open(fpath, 'rb') as f:
        data = pickle.load(f)
    print(f'{label}: {len(data)} entries reloaded from {fname}')